In [2]:
import numpy as np
import pandas as pd
from pulp import *
import copy

In [3]:
data = pd.read_csv('dataset.csv')
data = data.sort_values(by = 'Unnamed: 0')
ids = data[['Unnamed: 0']]
data = data.drop('Unnamed: 0', axis = 'columns')
data = data.reset_index(drop = True)
data.index = data.index+1
data.index = 'a'+ data.index.astype('str')
data

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
a1,3,840000.0,13.8,3.0,2.0,2.0,534.0,136.00,1965.0
a2,4,1100000.0,5.9,4.0,2.0,5.0,559.0,195.00,1920.0
a3,2,495000.0,13.9,2.0,1.0,1.0,76.0,79.00,1980.0
a4,3,1120000.0,7.8,3.0,2.0,1.0,293.0,180.00,2006.0
a5,4,2325000.0,9.2,4.0,3.0,2.0,638.0,314.00,1930.0
a6,3,822000.0,13.0,3.0,1.0,4.0,700.0,105.00,1950.0
a7,3,1560000.0,4.6,3.0,2.0,0.0,198.0,148.00,1910.0
a8,2,650000.0,10.5,2.0,1.0,3.0,620.0,85.00,1950.0
a9,3,1223500.0,7.9,3.0,2.0,2.0,721.0,136.00,1980.0
a10,2,790000.0,11.2,2.0,1.0,2.0,196.0,109.00,1970.0


In [3]:
data.describe()

,Rooms,Price,Distance,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt
count,50.000000,5.000000e+01,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000
mean,2.940000,1.010540e+06,10.760000,2.920000,1.580000,1.72000,400.440000,146.037400,1966.520000
std,0.977502,4.836915e+05,7.117727,0.965528,0.641745,1.03095,305.460586,67.062725,33.749522
min,1.000000,3.610000e+05,2.100000,1.000000,1.000000,0.00000,0.000000,46.000000,1890.000000
25%,2.000000,6.507500e+05,7.325000,2.000000,1.000000,1.00000,170.000000,105.250000,1942.500000
50%,3.000000,8.785000e+05,9.350000,3.000000,1.500000,2.00000,318.500000,132.000000,1970.000000
75%,3.000000,1.255000e+06,13.975000,3.000000,2.000000,2.00000,599.250000,183.750000,1992.750000
max,6.000000,2.440000e+06,45.900000,6.000000,3.000000,5.00000,1452.000000,314.000000,2013.000000


## **1. UTA algorithm - finding minimal subset of inconsistent constraints** ##

In [7]:
# preference_list - list of pairs (better, worse)
preference_list = [("a4", "a9"), ("a9", "a13"), ("a13", "a4"), ("a4", "a1"), ("a32", "a45"), 
("a27", "a18"), ("a25", "a28"), ("a45", "a50"), ("a43", "a37"), ("a39", "a38"), 
("a27", "a18"), ("a31", "a33"), ("a30", "a42"), ("a40", "a35"), ("a27", "a26"),
("a3", "a11"), ("a6", "a7")]

In [8]:
# criteria_types - list of pairs in form (gain or cost type, is discrete?)
criteria_types = [('gain', True), ('cost', False), 
('cost', False), ('gain', True), ('gain', True), 
('gain', True), ('gain', False), ('gain', False), 
('gain', False)]

In [9]:
def build_uta_model(preference_list, df, criteria_types):
    # 0. Define Linear Problem - we want to minimize number of const that are excluded
    model = LpProblem('uta_model', LpMinimize)
    eps = 0.001

    pairwise_comparisons = copy.deepcopy(preference_list)
    binary_variables = LpVariable.dicts('bin_var', pairwise_comparisons, cat='Binary')
    model += lpSum(list(binary_variables.values()))

    # 1. Add weights const
    weights = []
    for crit in df.columns:
        weights.append(LpVariable(f'w_{crit}', lowBound = 1/(5*len(df.columns)), upBound = 0.5, cat = 'Continuous'))

    # 2. Add norm const
    model += lpSum(weights) == 1

    # 3. Utility constraints
    utility_vars = {}
    for alt in df.index:
        for criterion in df.columns:
            curr_util_name = 'u_'+alt+'_'+criterion
            utility_vars[curr_util_name] = LpVariable(curr_util_name, lowBound=0, cat = 'Continuous')
    
    for i in range(len(criteria_types)):
        # Find best and worst
        if criteria_types[i][0] == 'cost':
            best_idx = np.argmin(df[df.columns[i]])
            worst_idx = np.argmax(df[df.columns[i]])
        else:
            worst_idx = np.argmin(df[df.columns[i]])
            best_idx = np.argmax(df[df.columns[i]])

        model += utility_vars[f'u_{df.index[best_idx]}_{df.columns[i]}'] == 1
        model += utility_vars[f'u_{df.index[worst_idx]}_{df.columns[i]}'] == 0

        # Monocity constraint
        if criteria_types[i][0] == 'cost':
            sorted_alts = np.argsort(df[df.columns[i]])
        else:
            sorted_alts = np.argsort(df[df.columns[i]])[::-1]
        for j in range(len(sorted_alts)-1):
                model += utility_vars[f'u_{df.index[sorted_alts[j]]}_{df.columns[i]}'] >= utility_vars[f'u_{df.index[sorted_alts[j+1]]}_{df.columns[i]}']
    print(model)

    # 4. Add utility functions
    for better, worse in pairwise_comparisons:
        # if criterion can't be met, then 1 is substracted on the right side so that it holds
        model += lpSum([weights[i] * utility_vars[f'u_{better}_{df.columns[i]}'] for i in range(len(weights))]) > lpSum([weights[i] * utility_vars[f'u_{worse}_{df.columns[i]}'] for i in range(len(weights))]) - binary_vars[(alt1, alt2)] + eps
    print(model)
    return model

In [22]:
def build_uta_model_2_1(preference_list, df, criteria_types):
    model = LpProblem('uta_model', LpMinimize)
    eps = 0.001

    # 0. Add binary variables for each pairwise preference
    pairwise_comparisons = copy.deepcopy(preference_list)
    binary_variables = LpVariable.dicts('bin_var', pairwise_comparisons, cat='Binary')
    model += lpSum(list(binary_variables.values()))

    # 1. Add weights
    weights = {}
    n_criteria = len(df.columns)
    for crit in df.columns:
        weights[crit] = LpVariable(f'w_{crit}', lowBound=1/(5*n_criteria), upBound=0.5, cat='Continuous')

    # 2. Add normalization constraint
    model += lpSum(weights.values()) == 1

    # 3. Utility values for each alternative per criterion (constants here)
    # Normalize criterion values between 0 and 1
    normalized_df = df.copy()
    for crit, (ctype, isDiscrete) in zip(df.columns, criteria_types):
        if ctype == 'cost':
            normalized_df[crit] = (df[crit].max() - df[crit]) / (df[crit].max() - df[crit].min())
        else:
            normalized_df[crit] = (df[crit] - df[crit].min()) / (df[crit].max() - df[crit].min())

    # Total utility for each alternative
    U = {}
    for alt in df.index:
        U[alt] = LpVariable(f'U_{alt}', lowBound=0, upBound=1, cat='Continuous')
        model += U[alt] == lpSum([weights[crit] * normalized_df.loc[alt, crit] for crit in df.columns])

    # 4. Pairwise constraints (some may be rejected by binary variables)
    for (alt1, alt2) in pairwise_comparisons:
        model += U[alt1] >= U[alt2] + eps - binary_variables[(alt1, alt2)]

    return model

In [23]:
model = build_uta_model_2_1(preference_list, data, criteria_types)
status = model.solve(solver = GLPK())

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --cpxlp /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/eac6c168e32f4e9f845e173985065817-pulp.lp
 -o /var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/eac6c168e32f4e9f845e173985065817-pulp.sol
Reading problem data from '/var/folders/8m/nx_b_wh17dg77b9kb95gxng80000gp/T/eac6c168e32f4e9f845e173985065817-pulp.lp'...
68 rows, 75 columns, 516 non-zeros
16 integer variables, all of which are binary
255 lines were read
GLPK Integer Optimizer 5.0
68 rows, 75 columns, 516 non-zeros
16 integer variables, all of which are binary
Preprocessing...
68 rows, 51 columns, 492 non-zeros
16 integer variables, all of which are binary
Scaling...
 A: min|aij| =  2.612e-02  max|aij| =  1.000e+00  ratio =  3.829e+01
GM: min|aij| =  1.950e-01  max|aij| =  5.129e+00  ratio =  2.631e+01
EQ: min|aij| =  3.801e-02  max|aij| =  1.000e+00  ratio =  2.631e+01
2N: min|aij| =  2.612e-02  max|aij| =  1.678e+00  ratio =  6.423e+01
Constructing ini

In [24]:
print("status:", model.status, LpStatus[model.status])
print("objective:", model.objective.value())

status: 1 Optimal
objective: 2


In [25]:
for var in model.variables():
    if 'bin_var' in var.name and var.value() > 0:
        print(var.name, var.value(), "is inconsistent")
    else:
        print(var.name, var.value())

U_a1 0.658863
U_a10 0.630897
U_a11 0.580312
U_a12 0.434803
U_a13 0.31916
U_a14 0.634154
U_a15 0.741041
U_a16 0.698714
U_a17 0.249192
U_a18 0.433738
U_a19 0.41602
U_a2 0.520701
U_a20 0.693387
U_a21 0.793467
U_a22 0.659264
U_a23 0.504655
U_a24 0.553935
U_a25 0.740246
U_a26 0.576896
U_a27 0.67859
U_a28 0.689672
U_a29 0.38003
U_a3 0.715514
U_a30 0.504889
U_a31 0.783493
U_a32 0.771356
U_a33 0.653766
U_a34 0.554252
U_a35 0.781855
U_a36 0.573615
U_a37 0.726642
U_a38 0.610872
U_a39 0.741852
U_a4 0.687696
U_a40 0.78502
U_a41 0.703222
U_a42 0.43938
U_a43 0.727642
U_a44 0.692066
U_a45 0.848526
U_a46 0.443439
U_a47 0.655186
U_a48 0.579996
U_a49 0.54069
U_a5 0.282879
U_a50 0.649095
U_a6 0.599831
U_a7 0.34639
U_a8 0.626249
U_a9 0.608198
bin_var_('a13',_'a4') 1 is inconsistent
bin_var_('a25',_'a28') 0
bin_var_('a27',_'a18') 0
bin_var_('a27',_'a26') 0
bin_var_('a3',_'a11') 0
bin_var_('a30',_'a42') 0
bin_var_('a31',_'a33') 0
bin_var_('a32',_'a45') 1 is inconsistent
bin_var_('a39',_'a38') 0
bin_var_('a4

## **UTA 2.2** ##

In [26]:
consistent_preference_list = list(set(preference_list).difference(('a13', 'a4'), ('a32', 'a45')))

In [27]:
consistent_preference_list

[('a31', 'a33'),
 ('a40', 'a35'),
 ('a43', 'a37'),
 ('a39', 'a38'),
 ('a3', 'a11'),
 ('a45', 'a50'),
 ('a30', 'a42'),
 ('a9', 'a13'),
 ('a4', 'a1'),
 ('a27', 'a18'),
 ('a6', 'a7'),
 ('a27', 'a26'),
 ('a32', 'a45'),
 ('a13', 'a4'),
 ('a25', 'a28'),
 ('a4', 'a9')]

In [ ]:
def build_uta_model_2_2(preference_list, df, criteria_types):
    